In [ ]:
### CS231n_Lec06_review_CNN_Architectures


import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

# fix Random number generation seed and the number of threads for CPU
torch.manual_seed(6)
torch.set_num_threads(1)

IMAGE_SIZE = 16
NUM_CLASSES = 3
CHANNELS = 16
NUM_BLOCKS = 8
EPOCHS = 12
BATCH_SIZE = 64

CLASS_NAMES = ("square", "plus", "X")



# --------------------------------------------------
# 1. Generate synthetic images with meaningful labels
# --------------------------------------------------
def make_dataset(num_samples, seed):
    """
    class 0: Square border
    class 1: + shape
    class 2: X shape

    The center position of the shape and the pixel noise change randomly.
    """
    generator = torch.Generator().manual_seed(seed)

    # Generate image space full of zero with IMAGE_SIZE×IMAGE_SIZE
    images = torch.zeros(
        num_samples, 1, IMAGE_SIZE, IMAGE_SIZE
    )

    labels = torch.randint(
        0,
        NUM_CLASSES,
        (num_samples,),
        generator=generator,
    )

    for i in range(num_samples):
        label = labels[i].item()

        # Select a random row and column within the range near the center
        center_row = torch.randint(
            4, 12, (), generator=generator
        ).item()

        center_column = torch.randint(
            4, 12, (), generator=generator
        ).item()

        offsets = []

        if label == 0:
            # 5×5 Square Border
            for offset in range(-2, 3):
                offsets.extend([
                    (-2, offset),
                    (2, offset),
                    (offset, -2),
                    (offset, 2),
                ])

        elif label == 1:
            # + shape
            for offset in range(-2, 3):
                offsets.extend([
                    (offset, 0),
                    (0, offset),
                ])

        else:
            # X shape
            for offset in range(-2, 3):
                offsets.extend([
                    (offset, offset),
                    (offset, -offset),
                ])

        # Assign Value '1' to the center row/column and the offset pixels.
        for row_offset, column_offset in offsets:
            row = center_row + row_offset
            column = center_column + column_offset
            images[i, 0, row, column] = 1.0

    # Add random noises
    noise = 0.15 * torch.randn(
        images.shape,
        generator=generator,
    )

    # the value of pixels is in (0,1) by .clamp
    images = (images + noise).clamp(0.0, 1.0)

    return images, labels


train_images, train_labels = make_dataset(900, seed=1)
test_images, test_labels = make_dataset(300, seed=2)

print("Train:", tuple(train_images.shape))
print("Test: ", tuple(test_images.shape))



# --------------------------------------------------
# 2. Plain Block and Residual Block
# --------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, channels, use_residual):
        super().__init__()

        # decide whether to use a residual connection
        self.use_residual = use_residual

        """
        flow: Conv2d -> BatchNorm2d → ReLU
        Note:
        1. the number of input channels = the number of output channels
        2. Height and width remain the same as the input
        for setting kernel_size and padding when using nn.Conv2d
        """
        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(channels)

        self.relu = nn.ReLU()

    def forward(self, x):

        # identity for a residual connection
        identity = x

        # input → conv1 → bn1 → conv2 → bn2 → (residual) → relu → output
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.use_residual:
            out = out + identity

        return self.relu(out)


class DeepCNN(nn.Module):
    def __init__(self, use_residual):
        super().__init__()

        # stem → The very front part of a CNN
        self.stem = nn.Sequential(
            nn.Conv2d(
                1,
                CHANNELS,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(CHANNELS),
            nn.ReLU(),
        )

        """
        About self.blocks:
        1. Implement a significantly deep CNN architecture using def ConvBlock.
        2. Create CNN layers by ConvBlock layers as the number of NUM_BLOCKS
           and combine them with the stem layer.
        3. '*'(astra) → Serves to unpack layer arguments
        """
        self.blocks = nn.Sequential(*[
            ConvBlock(CHANNELS, use_residual)
            for _ in range(NUM_BLOCKS)
        ])

        """
        remind for Lec05:
        While max pooling is useful for object classification,
        it can obscure precise positional information.
        """
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(CHANNELS, NUM_CLASSES)

    # flow: stem → block → pooling → flatten → classifier(output)
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)


# --------------------------------------------------
# 3. Measure Gradient Flow Before Training
# --------------------------------------------------
criterion = nn.CrossEntropyLoss()


##
def measure_gradients(model):
    # Set train mode + initialize gradients
    model.train()
    model.zero_grad()

    sample_x = train_images[:64]
    sample_y = train_labels[:64]

    loss = criterion(model(sample_x), sample_y)
    loss.backward()

    first_gradient = (
        model.stem[0].weight.grad.norm().item()
    )

    last_gradient = (
        model.blocks[-1].conv2.weight.grad.norm().item()
    )

    return first_gradient, last_gradient


# --------------------------------------------------
# 4. Model Training
# --------------------------------------------------
def train_model(use_residual):
    """
    # Ensure that both models have the same initial conditions
    as much as possible.
    """
    torch.manual_seed(20)

    model = DeepCNN(use_residual)

    first_grad, last_grad = measure_gradients(model)

    generator = torch.Generator().manual_seed(30)

    loader = DataLoader(
        TensorDataset(train_images, train_labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
    )

    loss_history = []
    test_accuracy_history = []

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0

        for batch_x, batch_y in loader:
            scores = model(batch_x)
            loss = criterion(scores, batch_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += (
                loss.item() * batch_x.size(0)
            )

        mean_loss = epoch_loss / len(loader.dataset)

        model.eval()

        with torch.no_grad():
            test_scores = model(test_images)

            test_accuracy = (
                test_scores.argmax(dim=1) == test_labels
            ).float().mean().item()

        loss_history.append(mean_loss)
        test_accuracy_history.append(test_accuracy)

    return {
        "model": model,
        "loss": loss_history,
        "test_accuracy": test_accuracy_history,
        "first_gradient": first_grad,
        "last_gradient": last_grad,
    }


plain_result = train_model(use_residual=False)
residual_result = train_model(use_residual=True)


# --------------------------------------------------
# 5. Output the model structure and final results
# --------------------------------------------------
def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


for name, result in (
    ("Plain", plain_result),
    ("Residual", residual_result),
):
    ratio = (
        result["first_gradient"]
        / (result["last_gradient"] + 1e-12)
    )

    print(
        f"{name:8s} | "
        f"parameters={count_parameters(result['model'])} | "
        f"test acc={result['test_accuracy'][-1]:.3f} | "
        f"first grad={result['first_gradient']:.3e} | "
        f"last grad={result['last_gradient']:.3e} | "
        f"first/last={ratio:.3f}"
    )


# --------------------------------------------------
# 6. Verification of tensor shapes within the Residual CNN
# --------------------------------------------------
resnet = residual_result["model"]
example_batch = test_images[:8]

with torch.no_grad():
    stem_output = resnet.stem(example_batch)
    block_output = resnet.blocks(stem_output)
    pooled = resnet.pool(block_output)
    scores = resnet(example_batch)

print("\n[Residual CNN tensor shapes]")
print("Input:       ", tuple(example_batch.shape))
print("Stem:        ", tuple(stem_output.shape))
print("Blocks:      ", tuple(block_output.shape))
print("Global pool: ", tuple(pooled.shape))
print("Scores:      ", tuple(scores.shape))

assert stem_output.shape == (8, CHANNELS, 16, 16)
assert block_output.shape == (8, CHANNELS, 16, 16)
assert pooled.shape == (8, CHANNELS, 1, 1)
assert scores.shape == (8, NUM_CLASSES)
assert torch.isfinite(scores).all()


# --------------------------------------------------
# 7. 학습 곡선 시각화
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(
    plain_result["loss"],
    label="Plain CNN",
)
axes[0].plot(
    residual_result["loss"],
    label="Residual CNN",
)
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(
    plain_result["test_accuracy"],
    label="Plain CNN",
)
axes[1].plot(
    residual_result["test_accuracy"],
    label="Residual CNN",
)
axes[1].set_title("Test accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()